# DM1 Project 2026/27 — Modulo 1: Data Understanding & Preparation

**Dataset**: *Seller-Level Vehicle Dataset* (`seller_info_dm1_2627.csv`) · **Target**: `seller_rating_class`

Notebook completo e autonomo per la parte 1 (30 punti). Segue la scaletta in `docs/Scaletta_Parte1_DU_DP.md`.

| Sezione | Blocco delle guideline (slide 7) |
|---|---|
| 1. Setup e caricamento | — |
| 2. Data semantics | *Data Semantics* |
| 3. Riparazione sintattica | *Assessing data quality* (accuratezza sintattica) |
| 4. Distribuzioni e statistiche | *Distribution of the variables and statistics* |
| 5. Qualità dei dati | *Assessing data quality* |
| 6. Valori mancanti | *Assessing data quality* |
| 7. Outlier, trasformazioni, nuove feature | *Possible variable transformation* |
| 8. Correlazioni ed eliminazione di variabili | *Pairwise correlations and eventual elimination* |
| 9. Dataset finali e riepilogo decisioni | ponte verso Clustering / Classificazione / Pattern Mining |

> **Nota sull'ordine.** Nel notebook la riparazione dei numeri corrotti (§3) viene **prima** delle statistiche (§4): senza di essa 7 colonne numeriche sarebbero stringhe e resterebbero fuori dalle analisi. Nel report l'ordine può seguire le guideline.
>
> **Convenzione.** Le celle *"Commento per il report"* contengono l'interpretazione dei risultati: rileggile dopo ogni esecuzione e adattale se i numeri cambiano.

## 1. Setup e caricamento

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from scipy.stats import chi2_contingency, spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from IPython.display import display

# Percorsi: funziona sia lanciando da notebooks/ sia dalla radice del progetto
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import dm1_prep as dp

DATA_PATH = ROOT / "data" / "seller_info_dm1_2627.csv"
FIG_DIR = ROOT / "report" / "figures"
TAB_DIR = ROOT / "report" / "tables"
OUT_DIR = ROOT / "data" / "processed"
for d in (FIG_DIR, TAB_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET, ORDER = dp.TARGET, dp.TARGET_ORDER

# Palette (validate per daltonismo):
# - target ordinale -> una sola tinta dal chiaro allo scuro (low -> high)
# - altre categoriche (max 3 serie) -> blu, arancio, acqua
# - una sola serie -> blu; heatmap di correlazione -> blu / grigio / rosso
SURFACE, INK, INK_2 = "#fcfcfb", "#0b0b0b", "#52514e"
BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"
CLASS_COLORS = {"low": "#86b6ef", "medium": "#2a78d6", "high": "#104281"}
CAT3 = [BLUE, ORANGE, AQUA]
DIVERGING = LinearSegmentedColormap.from_list("div", [BLUE, "#f0efec", RED])
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#f0efec", "#86b6ef", "#2a78d6", "#104281"])

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#c3c2b7", "grid.color": "#e6e5e0", "axes.titlesize": 11,
    "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.fontsize": 9, "text.color": INK, "axes.labelcolor": INK_2,
    "xtick.color": INK_2, "ytick.color": INK_2, "figure.dpi": 110,
})
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.3f}".format)
warnings.filterwarnings("ignore", category=FutureWarning)


def savefig(fig, name):
    dp.save_fig(fig, name, FIG_DIR)


def save_table(table, name):
    """Salva una tabella in CSV (e in LaTeX se jinja2 è disponibile) per il report."""
    table.to_csv(TAB_DIR / f"{name}.csv")
    try:
        table.to_latex(TAB_DIR / f"{name}.tex", float_format="%.2f")
    except ImportError:
        pass

In [ ]:
df_raw = dp.load_raw(DATA_PATH)          # separatore ';' (con il default si ottiene 1 sola colonna)
print("Shape:", df_raw.shape)
print("Righe duplicate:", df_raw.duplicated().sum(),
      "| sp_id duplicati:", df_raw["sp_id"].duplicated().sum(),
      "| duplicati ignorando sp_id:", df_raw.drop(columns="sp_id").duplicated().sum())
display(df_raw.head(3))

In [ ]:
# Tipi informatici assegnati da pandas: 7 colonne numeriche risultano stringhe
dtype_audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_NaN": df_raw.isna().sum(),
    "%NaN": (df_raw.isna().mean() * 100).round(2),
    "n_distinti": df_raw.nunique(),
    "esempio": df_raw.iloc[0],
})
display(dtype_audit)

**Commento per il report.** 13.843 venditori × 33 attributi, nessun duplicato (né di riga né di `sp_id`). Ogni riga è **un venditore**: i valori sono statistiche **aggregate** sui suoi annunci (mediane, proporzioni, mode). `sp_id` è letto come `float64` ma è un identificativo; 7 attributi numerici sono letti come stringhe a causa di valori corrotti (§3).

## 2. Data semantics

Classificazione di ogni attributo secondo le scale di Stevens (DM1.01 §3): **nominale** (solo =, ≠), **ordinale** (anche <, >), **intervallo** (anche differenze, zero arbitrario), **rapporto** (anche rapporti, zero assoluto).

Concetti chiave per questo dataset:
- le `*_median` sono **statistiche di secondo livello**: descrivono l'auto *tipica* dell'inventario, non una singola auto;
- le `pct_*` sono **proporzioni** in [0, 1];
- le `dominant_*` sono **mode** dell'inventario: un venditore con 51% SUV e 49% sedan risulta "suv" (si perde la varietà);
- `year_median` è su scala a **intervallo** (l'anno 0 è convenzionale), mentre `vehicle_age_median` è su scala a **rapporto**;
- il target è **ordinale** (low < medium < high) ed è la discretizzazione di un rating continuo che non ci viene fornito.

In [ ]:
SEMANTICS = [
    # (attributo, famiglia, significato, scala, discreto/continuo, ruolo, unità/dominio)
    ("sp_id", "Identificativo", "ID del venditore", "Nominale", "Discreto", "ID", "codice"),
    ("n_listings", "Attività", "N. totale di annunci", "Rapporto", "Discreto", "Predittore", "conteggio ≥ 5"),
    ("n_listing_days", "Attività", "Giorni distinti con annunci", "Rapporto", "Discreto", "Predittore", "giorni"),
    ("listing_span_days", "Attività", "Giorni tra primo e ultimo annuncio", "Rapporto", "Discreto", "Predittore", "giorni"),
    ("price_median", "Prezzo e mercato", "Prezzo mediano", "Rapporto", "Continuo", "Predittore", "USD"),
    ("savings_amount_median", "Prezzo e mercato", "Risparmio mediano pubblicizzato", "Rapporto", "Continuo", "Predittore", "USD"),
    ("daysonmarket_median", "Prezzo e mercato", "Giorni mediani di permanenza sul mercato", "Rapporto", "Continuo", "Predittore", "giorni"),
    ("mileage_median", "Età e uso", "Chilometraggio mediano", "Rapporto", "Continuo", "Predittore", "miglia"),
    ("year_median", "Età e uso", "Anno-modello mediano", "Intervallo", "Discreto", "Predittore", "anno"),
    ("vehicle_age_median", "Età e uso", "Età mediana all'annuncio", "Rapporto", "Continuo", "Predittore", "anni"),
    ("owner_count_median", "Età e uso", "N. mediano di proprietari precedenti", "Rapporto", "Discreto", "Predittore", "conteggio"),
    ("horsepower_median", "Motore e consumi", "Potenza mediana", "Rapporto", "Continuo", "Predittore", "hp"),
    ("engine_displacement_median", "Motore e consumi", "Cilindrata mediana", "Rapporto", "Continuo", "Predittore", "cc"),
    ("city_fuel_economy_median", "Motore e consumi", "Consumo urbano mediano", "Rapporto", "Continuo", "Predittore", "MPG"),
    ("highway_fuel_economy_median", "Motore e consumi", "Consumo extraurbano mediano", "Rapporto", "Continuo", "Predittore", "MPG"),
    ("back_legroom_median", "Dimensioni", "Spazio gambe posteriore", "Rapporto", "Continuo", "Predittore", "pollici"),
    ("front_legroom_median", "Dimensioni", "Spazio gambe anteriore", "Rapporto", "Continuo", "Predittore", "pollici"),
    ("fuel_tank_volume_median", "Dimensioni", "Capacità del serbatoio", "Rapporto", "Continuo", "Predittore", "galloni"),
    ("height_median", "Dimensioni", "Altezza", "Rapporto", "Continuo", "Predittore", "pollici"),
    ("length_median", "Dimensioni", "Lunghezza", "Rapporto", "Continuo", "Predittore", "pollici"),
    ("width_median", "Dimensioni", "Larghezza", "Rapporto", "Continuo", "Predittore", "pollici"),
    ("pct_isCab", "Composizione inventario", "Quota di ex taxi", "Rapporto", "Continuo", "Predittore", "[0, 1]"),
    ("pct_is_new", "Composizione inventario", "Quota di auto nuove", "Rapporto", "Continuo", "Predittore", "[0, 1]"),
    ("pct_salvage", "Composizione inventario", "Quota di auto incidentate (salvage)", "Rapporto", "Continuo", "Predittore", "[0, 1]"),
    ("pct_theft_title", "Composizione inventario", "Quota con titolo di furto", "Rapporto", "Continuo", "Predittore", "[0, 1]"),
    ("dominant_body_type", "Mode inventario", "Carrozzeria più frequente", "Nominale", "Discreto", "Predittore", "9 categorie"),
    ("dominant_fuel_type", "Mode inventario", "Alimentazione più frequente", "Nominale", "Discreto", "Predittore", "7 categorie"),
    ("dominant_transmission", "Mode inventario", "Cambio più frequente", "Nominale", "Discreto", "Predittore", "4 categorie"),
    ("dominant_wheel_system", "Mode inventario", "Trazione più frequente", "Nominale", "Discreto", "Predittore", "5 categorie"),
    ("dominant_listing_color", "Mode inventario", "Colore più frequente", "Nominale", "Discreto", "Predittore", "11 categorie"),
    ("franchise_status", "Profilo venditore", "Concessionario ufficiale o indipendente", "Nominale", "Discreto", "Predittore", "2 categorie"),
    ("new_used_profile", "Profilo venditore", "Composizione nuovo/usato (da pct_is_new)", "Nominale", "Discreto", "Predittore (derivata)", "3 categorie"),
    (TARGET, "Target", "Classe di rating del venditore", "Ordinale", "Discreto", "Target", "low < medium < high"),
]
semantics = pd.DataFrame(SEMANTICS, columns=["attributo", "famiglia", "significato", "scala",
                                             "discreto/continuo", "ruolo", "unità/dominio"]).set_index("attributo")
assert set(semantics.index) == set(df_raw.columns)
display(semantics)
display(semantics["scala"].value_counts().to_frame("n attributi"))
save_table(semantics, "tab_semantica")

In [ ]:
# Domini dichiarati nella documentazione vs valori osservati
DOC_DOMAINS = {
    "franchise_status": {"franchise", "independent", "mixed", "unknown"},
    "new_used_profile": {"new_only", "used_only", "mixed", "unknown"},
    TARGET: {"low", "medium", "high"},
}
rows = []
for col, declared in DOC_DOMAINS.items():
    observed = set(df_raw[col].dropna().unique())
    rows.append({"attributo": col,
                 "dichiarati ma assenti": sorted(declared - observed),
                 "presenti ma non dichiarati": sorted(observed - declared)})
display(pd.DataFrame(rows).set_index("attributo"))

# new_used_profile è una discretizzazione deterministica di pct_is_new?
pct_new_bin = pd.cut(df_raw["pct_is_new"], [-0.01, 0, 0.999999, 1], labels=["= 0", "(0, 1)", "= 1"])
display(pd.crosstab(df_raw["new_used_profile"], pct_new_bin))

# Anno di annuncio implicito: year_median + vehicle_age_median
listing_year = df_raw["year_median"] + df_raw["vehicle_age_median"]
print("Anno di annuncio implicito:", listing_year.describe()[["mean", "std", "min", "max"]].round(2).to_dict())

**Commento per il report.**
- La documentazione elenca 4 valori per `franchise_status` e `new_used_profile`, ma nei dati **"mixed" e "unknown" di `franchise_status` non compaiono** e **"unknown" di `new_used_profile` non compare**: c'è una discrepanza tra documentazione e dati.
- `new_used_profile` è **perfettamente determinata** da `pct_is_new` (0 → used_only, 1 → new_only, altrimenti mixed): è un attributo derivato, quindi ridondante (§8).
- `year_median + vehicle_age_median ≈ 2020` (std ≈ 0.4; esattamente 2020 per circa il 90% dei venditori): gli annunci sono stati raccolti intorno al 2020. Questo dà il contesto temporale (*timeliness*) e rende le due variabili ridondanti per costruzione. Il piccolo scarto residuo (§5.3) è normale: la somma di due mediane non coincide con la mediana della somma, e gli annunci di un venditore coprono più anni.

## 3. Riparazione sintattica (numeri corrotti)

In 7 colonne alcuni valori sono stringhe come `38.650.000.000.000.000` (legroom) o `6.619.999.999.999.990` (altezza): numeri decimali **rovinati da un'esportazione che ha trattato il punto come separatore delle migliaia** (38.65 → `38.650.000.000.000.000`).

Opzioni:
1. convertirli in NaN e poi imputarli → semplice, ma si butta via un'informazione che c'è;
2. **ricostruirli**: si tolgono i punti, si prova ogni posizione del punto decimale e si tiene l'unico candidato che cade nel range plausibile della colonna (0.1°–99.9° percentile dei valori leggibili);
3. **ibrida (scelta)**: ricostruzione quando il candidato plausibile è uno solo, NaN quando ce ne sono zero o più di uno (non si inventa nulla).

In [ ]:
df = df_raw.copy()
repair_report, corrupted_mask = {}, {}
for col in dp.CORRUPTED_COLS:
    df[col], repair_report[col], corrupted_mask[col] = dp.repair_corrupted_numeric(df_raw[col])
repair_table = pd.DataFrame(repair_report).T
display(repair_table)
save_table(repair_table, "tab_riparazione_numeri")

# Alcuni esempi: valore grezzo -> valore ricostruito
examples = pd.concat([
    pd.DataFrame({"colonna": c, "grezzo": df_raw.loc[corrupted_mask[c], c],
                  "ricostruito": df.loc[corrupted_mask[c], c]}).head(2)
    for c in dp.CORRUPTED_COLS
])
display(examples)

In [ ]:
# Verifica: i valori ricostruiti devono avere la stessa distribuzione dei valori leggibili
cols_check = ["back_legroom_median", "height_median", "width_median", "fuel_tank_volume_median"]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, col in zip(axes, cols_check):
    clean_vals = df.loc[~corrupted_mask[col], col].dropna()
    fixed_vals = df.loc[corrupted_mask[col], col].dropna()
    bins = np.histogram_bin_edges(clean_vals, bins=40)
    ax.hist(clean_vals, bins=bins, density=True, histtype="step", lw=2, color=BLUE,
            label=f"leggibili (n={len(clean_vals):,})")
    ax.hist(fixed_vals, bins=bins, density=True, histtype="step", lw=2, ls="--", color=ORANGE,
            label=f"ricostruiti (n={len(fixed_vals):,})")
    ax.set_title(col)
    ax.set_ylabel("densità")
    ax.legend(frameon=False, fontsize=8)
fig.suptitle("Valori ricostruiti vs valori leggibili", y=1.03)
savefig(fig, "fig03_riparazione_numeri")
plt.show()

In [ ]:
# Pulizia dei tipi: sp_id come indice intero, codici del cambio leggibili
df["sp_id"] = df["sp_id"].astype("int64")
df = df.set_index("sp_id")
df["dominant_transmission"] = df["dominant_transmission"].replace({"a": "automatic", "m": "manual"})
print(df.dtypes.value_counts())

**Commento per il report.** Circa 1.580 celle corrotte; il **99,7%** è stato ricostruito in modo univoco. Le distribuzioni dei valori ricostruiti si sovrappongono a quelle dei valori leggibili (figura), quindi la ricostruzione non introduce distorsioni. I pochi casi ambigui o senza candidato (2 prezzi, 3 misure) diventano NaN e vengono trattati in §6.

## 4. Distribuzioni e statistiche

### 4.1 Statistiche descrittive (numeriche)
Il confronto **media vs mediana** e la **skewness** indicano l'asimmetria; `n_zeri` mette in luce le distribuzioni con molta massa in 0 e i possibili mancanti mascherati.

In [ ]:
GROUPS = {
    "Attività del venditore": ["n_listings", "n_listing_days", "listing_span_days"],
    "Prezzo e mercato": ["price_median", "savings_amount_median", "daysonmarket_median"],
    "Età e uso": ["mileage_median", "year_median", "vehicle_age_median", "owner_count_median"],
    "Motore e consumi": ["horsepower_median", "engine_displacement_median",
                         "city_fuel_economy_median", "highway_fuel_economy_median"],
    "Dimensioni": ["back_legroom_median", "front_legroom_median", "fuel_tank_volume_median",
                   "height_median", "length_median", "width_median"],
    "Composizione inventario": ["pct_isCab", "pct_is_new", "pct_salvage", "pct_theft_title"],
}
NUM = [c for cols in GROUPS.values() for c in cols]
CAT = ["dominant_body_type", "dominant_fuel_type", "dominant_transmission", "dominant_wheel_system",
       "dominant_listing_color", "franchise_status", "new_used_profile"]

stats = dp.describe_plus(df, NUM)
display(stats.round(2))
save_table(stats.round(2), "tab_statistiche_descrittive")

SKEWED = stats.index[stats["skew"].abs() > 2].tolist()
print("Variabili con |skew| > 2:", SKEWED)

# Nei grafici si usa il log solo per conteggi, durate e importi a coda lunga
# (non per year_median, che ha asimmetria negativa, né per proporzioni o misure fisiche)
PLOT_LOG = ["n_listings", "n_listing_days", "listing_span_days", "daysonmarket_median",
            "price_median", "savings_amount_median"]

### 4.2 Distribuzioni univariate delle numeriche
Istogrammi per famiglia di variabili, con media (tratteggio) e mediana (linea continua). Per leggibilità l'asse x è tagliato al 99,5° percentile: la coda estrema è visibile nei box plot (§4.3).

In [ ]:
def hist_grid(data, cols, title, fname, ncols=3, clip_q=0.995, bins=40):
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 2.9 * nrows), squeeze=False)
    axes = axes.ravel()
    for i, (ax, col) in enumerate(zip(axes, cols)):
        s = data[col].dropna()
        upper = s.quantile(clip_q)
        shown = s[s <= upper]
        ax.hist(shown, bins=bins, color=BLUE, edgecolor=SURFACE, linewidth=0.6)
        ax.axvline(s.mean(), color=INK, ls="--", lw=1.4, label="media")
        ax.axvline(s.median(), color=ORANGE, lw=2, label="mediana")
        ax.set_title(f"{col}\nskew = {s.skew():.2f}", fontsize=10)
        ax.set_ylabel("n venditori")
        if i == 0:
            ax.legend(frameon=False, fontsize=8)
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    savefig(fig, fname)
    plt.show()


GROUP_SLUGS = ["attivita", "mercato", "eta_uso", "motore", "dimensioni", "composizione"]   # nomi file senza accenti
for k, ((group, cols), slug) in enumerate(zip(GROUPS.items(), GROUP_SLUGS), start=1):
    hist_grid(df, cols, group, f"fig04_{k}_hist_{slug}")

### 4.3 Box plot (outlier visibili, scala completa)

In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(18, 10))
for ax, col in zip(axes.ravel(), NUM):
    ax.boxplot(df[col].dropna(), widths=0.5, patch_artist=True,
               boxprops=dict(facecolor="#cde2fb", edgecolor=BLUE),
               medianprops=dict(color=ORANGE, lw=2), whiskerprops=dict(color=BLUE),
               capprops=dict(color=BLUE),
               flierprops=dict(marker="o", markersize=2, markerfacecolor=INK_2, alpha=0.3, markeredgewidth=0))
    ax.set_title(col, fontsize=9)
    ax.set_xticks([])
fig.suptitle("Box plot delle variabili numeriche (baffi a 1.5·IQR)", y=1.0)
fig.tight_layout()
savefig(fig, "fig04_7_boxplot_numeriche")
plt.show()

**Commento per il report (§4.1–4.3).**
- **Forte asimmetria positiva** (media ≫ mediana): `n_listings` (mediana 70, max 4.959), `daysonmarket_median` (max 1.987), `listing_span_days`, `price_median` (max ~236k USD), `savings_amount_median`. Sono distribuzioni a coda lunga, tipiche di conteggi e prezzi → candidate al logaritmo (§7).
- **Asimmetria negativa** di `year_median`, dovuta a una piccola coda di concessionari di auto d'epoca (1960–1988).
- **Molta massa in 0**: `pct_salvage` (~91% zeri), `pct_theft_title` (~85%), `savings_amount_median` (~37%), `pct_is_new` (~53% = venditori di solo usato), `mileage_median` (621 venditori di solo nuovo).
- Variabili quasi costanti: `front_legroom_median` (std ≈ 0.8 pollici) → probabilmente poco informativa.
- I box plot segnalano molti punti oltre i baffi per le variabili asimmetriche: non sono necessariamente errori (§7.1).

### 4.4 Distribuzioni delle categoriche

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7.5))
for ax, col in zip(axes.ravel(), CAT + [TARGET]):
    counts = df[col].fillna("NaN (mancante)").value_counts()
    if col == TARGET:
        counts = counts.reindex(ORDER)
    pct = counts / counts.sum() * 100
    colors = [CLASS_COLORS[c] for c in counts.index] if col == TARGET else BLUE
    ax.barh(counts.index[::-1], counts.values[::-1],
            color=colors[::-1] if isinstance(colors, list) else colors, edgecolor=SURFACE, linewidth=1)
    for y, (v, p) in enumerate(zip(counts.values[::-1], pct.values[::-1])):
        ax.text(v, y, f" {p:.1f}%", va="center", fontsize=8, color=INK_2)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("n venditori")
    ax.set_xlim(0, counts.max() * 1.25)
fig.suptitle("Frequenze delle variabili categoriche e del target", y=1.0)
fig.tight_layout()
savefig(fig, "fig04_8_categoriche")
plt.show()

**Commento per il report.**
- **Target sbilanciato**: medium ~55%, high ~27%, low ~19% → in classificazione serviranno split stratificato e metriche come la macro-F1, non la sola accuratezza.
- `dominant_fuel_type` (gasoline ~98,7%) e `dominant_transmission` (automatico ~94%) sono **quasi costanti**: portano pochissima informazione.
- `dominant_body_type`: SUV ~64%, sedan ~22%, pickup ~10%, le altre 6 categorie sotto il 2% → da raggruppare (§7.3).
- `dominant_wheel_system`: **76% di NaN** (§6). `dominant_listing_color = "unknown"` (~11%) è un **mancante mascherato**.
- `franchise_status` è quasi perfettamente bilanciata (franchise 51%, independent 49%).

### 4.5 Bimodalità: venditori di nuovo vs di usato
La composizione dell'inventario divide i venditori in due popolazioni con età e chilometraggio molto diversi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
profiles = ["used_only", "mixed", "new_only"]
for ax, col in zip(axes, ["mileage_median", "vehicle_age_median"]):
    bins = np.histogram_bin_edges(df[col].dropna(), bins=40)
    for prof, color in zip(profiles, CAT3):
        s = df.loc[df["new_used_profile"] == prof, col].dropna()
        ax.hist(s, bins=bins, density=True, histtype="step", lw=2, color=color, label=f"{prof} (n={len(s):,})")
    ax.set_title(col)
    ax.set_ylabel("densità")
axes[0].legend(frameon=False)
fig.suptitle("Distribuzione per profilo nuovo/usato", y=1.02)
fig.tight_layout()
savefig(fig, "fig04_9_bimodalita_profilo")
plt.show()

### 4.6 Relazioni con il target
Prima una misura numerica (Spearman con il target codificato low=0, medium=1, high=2: è ordinale, quindi Spearman è adatto), poi i grafici sulle variabili più associate.

In [ ]:
target_code = df[TARGET].map({c: i for i, c in enumerate(ORDER)})
rho_target = pd.Series({c: spearmanr(df[c], target_code, nan_policy="omit")[0] for c in NUM}, name="rho_spearman")
rho_target = rho_target.reindex(rho_target.abs().sort_values(ascending=False).index)
display(rho_target.to_frame().round(3))
TOP_VS_TARGET = rho_target.index[:8].tolist()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, col in zip(axes.ravel(), TOP_VS_TARGET):
    y = np.log1p(df[col]) if col in PLOT_LOG else df[col]
    sns.boxplot(x=df[TARGET], y=y, order=ORDER, hue=df[TARGET], hue_order=ORDER, palette=CLASS_COLORS,
                showfliers=False, ax=ax, legend=False, linewidth=1.2)
    ax.set_title(f"{col}  (ρ = {rho_target[col]:.2f})", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel(f"log1p({col})" if col in PLOT_LOG else col, fontsize=8)
fig.suptitle("Variabili più associate al rating (box plot per classe, senza outlier)", y=1.0)
fig.tight_layout()
savefig(fig, "fig04_10_boxplot_vs_target")
plt.show()

In [ ]:
# KDE per classe: con classi sbilanciate serve common_norm=False
fig, axes = plt.subplots(2, 4, figsize=(17, 6.5))
for i, (ax, col) in enumerate(zip(axes.ravel(), TOP_VS_TARGET)):
    x = np.log1p(df[col]) if col in PLOT_LOG else df[col]
    sns.kdeplot(x=x, hue=df[TARGET], hue_order=ORDER, palette=CLASS_COLORS, common_norm=False,
                fill=False, linewidth=2, ax=ax, legend=(i == 0), warn_singular=False)
    ax.set_xlabel(f"log1p({col})" if col in PLOT_LOG else col, fontsize=8)
    ax.set_title(col, fontsize=10)
fig.suptitle("Densità per classe di rating", y=1.0)
fig.tight_layout()
savefig(fig, "fig04_11_kde_vs_target")
plt.show()

In [ ]:
# Categoriche vs target: P(rating | categoria), barre al 100%
cats_vs_target = ["franchise_status", "new_used_profile", "dominant_body_type", "dominant_listing_color"]
fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
for ax, col in zip(axes, cats_vs_target):
    counts = df[col].fillna("NaN").value_counts()
    ct = pd.crosstab(df[col].fillna("NaN"), df[TARGET], normalize="index")[ORDER]
    ct = ct.loc[counts.index]                                        # ordine per frequenza
    ct.index = [f"{c} (n={counts[c]:,})" for c in ct.index]          # numerosità: 100% su n=1 non vuol dire nulla
    left = np.zeros(len(ct))
    for cls in ORDER:
        ax.barh(ct.index[::-1], ct[cls].values[::-1], left=left[::-1], color=CLASS_COLORS[cls],
                edgecolor=SURFACE, linewidth=1.5, label=cls)
        for y, (l, v) in enumerate(zip(left[::-1], ct[cls].values[::-1])):
            if v >= 0.12:
                ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=7,
                        color="white" if cls != "low" else INK)
        left = left + ct[cls].values
    ax.set_title(col, fontsize=10)
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
fig.tight_layout(rect=(0, 0, 1, 0.88))
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title=TARGET, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 0.95), ncol=3)
fig.suptitle("Distribuzione del rating per categoria (n = numero di venditori)", y=1.0)
savefig(fig, "fig04_12_categoriche_vs_target")
plt.show()

for col in ["franchise_status", "new_used_profile"]:
    display(pd.crosstab(df[col], df[TARGET], normalize="index")[ORDER].round(3))

In [ ]:
# Pairplot su un sottoinsieme di variabili e un campione stratificato (leggibilità e velocità)
sample = df.groupby(TARGET, group_keys=False).sample(frac=0.2, random_state=RANDOM_STATE)
pair_cols = ["n_listings", "price_median", "mileage_median", "vehicle_age_median", "horsepower_median"]
pair_df = sample[pair_cols + [TARGET]].copy()
for c in ["n_listings", "price_median"]:
    pair_df[c] = np.log1p(pair_df[c])
pair_df = pair_df.rename(columns={"n_listings": "log1p(n_listings)", "price_median": "log1p(price_median)"})
g = sns.pairplot(pair_df, hue=TARGET, hue_order=ORDER, palette=CLASS_COLORS, corner=True,
                 plot_kws=dict(s=8, alpha=0.4, linewidth=0), diag_kws=dict(common_norm=False, fill=False))
g.figure.suptitle("Scatter matrix (campione stratificato del 20%)", y=1.01)
savefig(g.figure, "fig04_13_pairplot")
plt.show()

In [ ]:
# Coordinate parallele: 100 venditori per classe, variabili standardizzate (con log per le asimmetriche)
pc_cols = ["n_listings", "n_listing_days", "vehicle_age_median", "mileage_median", "pct_is_new",
           "savings_amount_median"]
pc = df[pc_cols + [TARGET]].dropna().copy()
for c in pc_cols:
    if c in PLOT_LOG:
        pc[c] = np.log1p(pc[c])
pc[pc_cols] = StandardScaler().fit_transform(pc[pc_cols])
pc_sample = pc.groupby(TARGET, group_keys=False).sample(n=100, random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(12, 4.5))
xs = np.arange(len(pc_cols))
for cls in ORDER:
    sub = pc_sample[pc_sample[TARGET] == cls]
    for _, row in sub[pc_cols].iterrows():
        ax.plot(xs, row.values, color=CLASS_COLORS[cls], alpha=0.08, lw=0.8)
for cls in ORDER:   # profilo mediano di ogni classe, calcolato su tutti i venditori
    ax.plot(xs, pc.loc[pc[TARGET] == cls, pc_cols].median().values, color=CLASS_COLORS[cls],
            lw=3, marker="o", markersize=8, label=f"mediana {cls}")
ax.set_xticks(xs)
ax.set_xticklabels([f"log1p({c})" if c in PLOT_LOG else c for c in pc_cols], rotation=15, fontsize=8)
ax.set_ylabel("valore standardizzato (z)")
ax.set_ylim(-3, 3)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.12))
ax.set_title("Coordinate parallele: 100 venditori per classe + profilo mediano", pad=28)
savefig(fig, "fig04_14_coordinate_parallele")
plt.show()

**Commento per il report (§4.5–4.6).**
- `mileage_median` e `vehicle_age_median` sono **bimodali**: i venditori `used_only` e `mixed` formano due popolazioni distinte (km ed età alti contro bassi). Il profilo nuovo/usato è la principale fonte di struttura del dataset, un dato utile da tenere a mente per il clustering.
- Le relazioni con il target sono **deboli**: la |ρ| di Spearman più alta è intorno a 0,26 (`n_listings`, `n_listing_days`, entrambe negative: **i venditori più grandi e attivi tendono ad avere rating più basso**). Le KDE delle tre classi si sovrappongono molto. È coerente con l'avviso delle guideline sul task di classificazione "difficile".
- `franchise_status`: gli **indipendenti** hanno più rating *high* (~33% contro ~21%) e meno *low* (~16% contro ~21%). `new_used_profile` mostra lo stesso schema (used_only ≈ independent, mixed ≈ franchise): le due variabili portano quasi la stessa informazione (§8.3).
- Nelle coordinate parallele le singole linee delle tre classi si mescolano completamente, ma i profili mediani mostrano tendenze coerenti con quanto visto sopra: la classe *high* ha meno annunci e meno giorni di attività e auto un po' più vecchie; la classe *low* ha la quota più alta di auto nuove (venditori grandi e misti, tipicamente concessionari ufficiali).
- Tra le categoriche, l'associazione più forte con il target è quella di `franchise_status` (V di Cramér ≈ 0,14, §8.2): comunque debole.

## 5. Qualità dei dati

### 5.1 Quadro per dimensioni della qualità (DM1.01 §4.2)

In [ ]:
pct_cols = GROUPS["Composizione inventario"]
quality = pd.DataFrame([
    ("Accuratezza sintattica", "valori nel formato/dominio corretto?",
     f"{int(repair_table['corrotti'].sum())} numeri corrotti in 7 colonne; sp_id letto come float; "
     f"pct_* fuori da [0,1]: {int(((df[pct_cols] < 0) | (df[pct_cols] > 1)).sum().sum())}",
     "ricostruzione (§3), cast dei tipi"),
    ("Accuratezza semantica", "valori plausibili nel mondo reale?",
     "legroom 0 o > 50, consumi > 60 MPG su non elettrici, mileage 0 su venditori di usato",
     "caso per caso (§5.2): errore → NaN, vero → si tiene"),
    ("Completezza", "quanto manca (anche mascherato)?",
     f"wheel_system {df['dominant_wheel_system'].isna().mean():.0%}, engine_displacement "
     f"{df['engine_displacement_median'].isna().mean():.1%}, colore 'unknown' "
     f"{(df['dominant_listing_color'] == 'unknown').mean():.0%}",
     "§6"),
    ("Coerenza", "attributi coerenti tra loro e con la documentazione?",
     "year + age ≈ 2020 (scarti piccoli e spiegabili); profilo ↔ pct_is_new coerenti; categorie documentate assenti",
     "§2, §5.3"),
    ("Sbilanciamento", "classi/categorie equilibrate?",
     "target 55/27/19%; gasoline 98.7%; automatico 94%",
     "stratificazione in classificazione; raggruppamento categorie rare (§7.3)"),
    ("Attualità (timeliness)", "a quando risalgono i dati?",
     "annunci intorno al 2020; 32 venditori con auto del 1960–1988",
     "dichiarare il periodo; le auto d'epoca sono una nicchia reale"),
    ("Unicità", "duplicati?", "0 righe e 0 sp_id duplicati", "nessuna"),
], columns=["dimensione", "domanda", "riscontro", "azione"]).set_index("dimensione")
display(quality)
save_table(quality, "tab_qualita_dimensioni")

### 5.2 Controlli di dominio (valori sospetti)
Per ogni gruppo di valori sospetti va deciso se è un **errore** (va corretto o reso NaN) o un **valore vero** (va tenuto). È il "dilemma" visto a lezione: errore e valore vero richiedono azioni opposte.

In [ ]:
city, hwy = df["city_fuel_economy_median"], df["highway_fuel_economy_median"]
is_ev = df["dominant_fuel_type"].eq("electric")
is_hybrid = df["dominant_fuel_type"].eq("hybrid")
DOMAIN_CHECKS = [
    ("back_legroom_median = 0", df["back_legroom_median"].eq(0),
     "dubbio", "pickup a cabina singola? 'non applicabile' → NaN"),
    ("front_legroom_median > 50", df["front_legroom_median"].gt(50),
     "errore", "fisicamente fuori scala → NaN"),
    ("height_median > 90", df["height_median"].gt(90), "vero", "van alti → si tengono"),
    ("city MPG > 60, elettrici", city.gt(60) & is_ev, "vero", "unità MPGe → si tengono"),
    ("city MPG > 60, non elettrici", city.gt(60) & ~is_ev, "dubbio", "probabili ibridi mal etichettati → NaN"),
    ("highway < city", hwy.lt(city), "vero", "tipico di ibridi/EV → si tengono"),
    ("year_median < 1990", df["year_median"].lt(1990), "vero", "concessionari di auto d'epoca → si tengono"),
    ("daysonmarket_median > 365", df["daysonmarket_median"].gt(365), "vero", "coda lunga → log (§7)"),
    ("listing_span_days > 1500", df["listing_span_days"].gt(1500), "vero", "coda lunga → log (§7)"),
    ("mileage = 0 ma pct_is_new < 0.5", df["mileage_median"].eq(0) & df["pct_is_new"].lt(0.5),
     "errore", "incoerente con un inventario di usato → NaN"),
]
domain_table = pd.DataFrame(
    [(name, int(mask.sum()), kind, action) for name, mask, kind, action in DOMAIN_CHECKS],
    columns=["controllo", "n venditori", "classificazione", "azione"]).set_index("controllo")
display(domain_table)
save_table(domain_table, "tab_controlli_dominio")
print("Hybrid/EV tra i casi 'highway < city':", int((hwy.lt(city) & (is_ev | is_hybrid)).sum()), "su", int(hwy.lt(city).sum()))

### 5.3 Coerenza tra variabili
Anche i controlli superati vanno riportati: mostrano che la verifica è stata fatta.

> Attenzione: il controllo `n_listing_days ≤ listing_span_days` sarebbe **sbagliato**. Un venditore con annunci in un solo giorno ha 1 giorno distinto e span 0. La condizione corretta è `listing_span_days ≥ n_listing_days − 1`.

In [ ]:
COHERENCE = {
    "listing_span_days ≥ n_listing_days − 1": df["listing_span_days"] >= df["n_listing_days"] - 1,
    "n_listing_days ≤ n_listings": df["n_listing_days"] <= df["n_listings"],
    "|year + age − 2020| ≤ 1": (df["year_median"] + df["vehicle_age_median"] - 2020).abs().le(1)
                               | df["vehicle_age_median"].isna(),
    "used_only ⇒ pct_is_new = 0": ~df["new_used_profile"].eq("used_only") | df["pct_is_new"].eq(0),
    "new_only ⇒ pct_is_new = 1": ~df["new_used_profile"].eq("new_only") | df["pct_is_new"].eq(1),
    "used_only ⇒ mileage > 0": ~df["new_used_profile"].eq("used_only") | df["mileage_median"].gt(0)
                               | df["mileage_median"].isna(),
}
coherence_table = pd.DataFrame({"violazioni": {k: int((~v).sum()) for k, v in COHERENCE.items()}})
coherence_table["% venditori"] = (coherence_table["violazioni"] / len(df) * 100).round(2)
coherence_table["esito"] = np.where(coherence_table["violazioni"] == 0, "OK", "da verificare")
display(coherence_table)
print("Con la condizione sbagliata (n_listing_days ≤ span):",
      int((df["n_listing_days"] > df["listing_span_days"]).sum()), "falsi errori")
print("Distribuzione di round(year + age):")
display((df["year_median"] + df["vehicle_age_median"]).round().value_counts().sort_index().to_frame("n venditori"))

In [ ]:
# Applicazione delle correzioni: errori e casi dubbi → NaN (le righe vengono tenute)
corrections = {
    "back_legroom_median": df["back_legroom_median"].eq(0),
    "front_legroom_median": df["front_legroom_median"].gt(50),
    "city_fuel_economy_median": city.gt(60) & ~is_ev,
    "highway_fuel_economy_median": city.gt(60) & ~is_ev,
    "mileage_median": df["mileage_median"].eq(0) & df["pct_is_new"].lt(0.5),
}
for col, mask in corrections.items():
    df.loc[mask, col] = np.nan
print({k: int(v.sum()) for k, v in corrections.items()})

**Commento per il report.** Le anomalie semantiche sono **poche** (decine di venditori su 13.843) e quasi tutte spiegabili: auto elettriche con consumi in MPGe, van alti, auto d'epoca e ibridi con consumo extraurbano peggiore di quello urbano sono **valori veri** e si tengono. Diventano NaN solo i valori fisicamente impossibili o incoerenti (legroom fuori scala, km nulli in inventari di usato) e i casi dubbi (2 "gasoline" a ~72 MPG).

I controlli di coerenza sono tutti superati tranne uno: per circa l'1,7% dei venditori `year + age` si allontana da 2020 di più di un anno (valori tra 2015 e 2024, quasi tutti 2019 o 2021–2022). **Non è un errore**: la somma di due mediane non coincide con la mediana della somma, e gli annunci di un venditore coprono più anni (`listing_span_days` arriva a ~10 anni). Il controllo conferma che le due variabili misurano la stessa cosa a meno di un piccolo scarto.

## 6. Valori mancanti

### 6.1 Quanti sono e come si distribuiscono

In [ ]:
na_table = pd.DataFrame({"n_NaN": df.isna().sum(), "%NaN": df.isna().mean() * 100})
na_table = na_table[na_table["n_NaN"] > 0].sort_values("n_NaN", ascending=False)
display(na_table.round(2))
unknown_counts = {c: int((df[c] == "unknown").sum()) for c in CAT}
print("Mancanti mascherati 'unknown':", {k: v for k, v in unknown_counts.items() if v})
save_table(na_table.round(2), "tab_mancanti")

In [ ]:
# Matrice dei mancanti: le righe sono ordinate per numero di NaN, per vedere se mancano "in blocco"
na_cols = na_table.index.tolist()
na_matrix = df[na_cols].isna()
order_rows = na_matrix.sum(axis=1).sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(na_matrix.loc[order_rows].T.values, aspect="auto", interpolation="nearest",
          cmap=ListedColormap(["#f0efec", BLUE]))
ax.set_yticks(range(len(na_cols)))
ax.set_yticklabels(na_cols, fontsize=8)
ax.set_xlabel("venditori (ordinati per n. di valori mancanti)")
ax.set_title("Mappa dei valori mancanti (blu = mancante)")
ax.grid(False)
savefig(fig, "fig06_1_mappa_mancanti")
plt.show()

block = df["pct_isCab"].isna()
print("Righe con pct_isCab NaN:", int(block.sum()))
print("Di queste, NaN anche in:", df.loc[block].isna().sum()[lambda s: s > 0].to_dict())
# Chi sono questi venditori?
print("Profilo nuovo/usato:", df.loc[block, "new_used_profile"].value_counts().to_dict(),
      "| pct_is_new min:", round(df.loc[block, "pct_is_new"].min(), 3),
      "| new_only totali nel dataset:", int(df["new_used_profile"].eq("new_only").sum()))
print("owner_count NaN fuori dal blocco, per profilo:",
      df.loc[df["owner_count_median"].isna() & ~block, "new_used_profile"].value_counts().to_dict())

### 6.2 Il fatto di mancare porta informazione?
Per le due variabili con molti mancanti si confrontano le righe con e senza mancante: (a) mediane delle altre variabili, (b) distribuzione del target con test χ² e V di Cramér. Il target viene usato **solo per descrivere**, mai per imputare.

In [ ]:
inform_cols = ["engine_displacement_median", "dominant_wheel_system"]
compare_cols = ["n_listings", "price_median", "vehicle_age_median", "horsepower_median",
                "fuel_tank_volume_median", "pct_is_new"]
for col in inform_cols:
    miss = df[col].isna().rename(f"{col} mancante")
    print(f"\n=== {col}: {miss.mean():.1%} mancanti ===")
    display(df.groupby(miss)[compare_cols].median().round(2))
    ct = pd.crosstab(miss, df[TARGET], normalize="index")[ORDER]
    chi2, p, *_ = chi2_contingency(pd.crosstab(miss, df[TARGET]))
    display(ct.round(3))
    print(f"chi2 = {chi2:.1f}, p = {p:.3g}, V di Cramér con il target = {dp.cramers_v(miss, df[TARGET]):.3f}")
    print("Associazione con franchise_status:", round(dp.cramers_v(miss, df["franchise_status"]), 3),
          "| con dominant_body_type:", round(dp.cramers_v(miss, df["dominant_body_type"]), 3))

**Come leggere il risultato.** Se le mediane e la distribuzione del target sono quasi uguali tra i due gruppi (V di Cramér ≈ 0), i mancanti sono "casuali" e si possono imputare o eliminare senza problemi. Se invece differiscono, il mancante è **informativo**: conviene tenerne traccia con una colonna binaria (`*_missing`) o con una categoria `unknown`.

> Con 13.843 righe anche differenze minime risultano "significative" nel test χ² (p piccolo): per giudicare conviene guardare la **dimensione dell'effetto** (V di Cramér, differenze tra mediane), non solo il p-value.

### 6.3 Mancanti strutturali: "non applicabile"
I 27 venditori con NaN in blocco su `pct_isCab`, `pct_salvage`, `pct_theft_title` e `owner_count_median` sono **quasi tutti venditori di solo nuovo** (26 dei 31 `new_only` del dataset). Per un'auto nuova queste grandezze **non si applicano**: non ha proprietari precedenti e non può essere un ex taxi, un'incidentata o un'auto rubata. Non è un dato perso, è un dato che non esiste (DM1.01 §14: *"l'attributo non si applica"*).

Scelta:
- **non eliminare le righe**: si perderebbe l'84% della categoria `new_only`;
- `pct_isCab`, `pct_salvage`, `pct_theft_title` → **0** (nessuna auto dell'inventario ha quella caratteristica);
- `owner_count_median` → **0** per i venditori con almeno il 90% di auto nuove (0 proprietari precedenti). È un valore nuovo nel dominio (prima il minimo era 1), ma coerente con il significato della variabile.

In [ ]:
df = df.drop(columns="dominant_wheel_system")       # 76% NaN: troppo vuota per essere imputata in modo credibile

assert df.loc[block, "pct_is_new"].ge(0.9).all()    # il blocco è fatto solo di inventari (quasi) tutti nuovi
structural_fill = {}
for col in ["pct_isCab", "pct_salvage", "pct_theft_title"]:
    structural_fill[col] = int(df[col].isna().sum())
    df.loc[df[col].isna(), col] = 0.0
owner_na_new = df["owner_count_median"].isna() & df["pct_is_new"].ge(0.9)
structural_fill["owner_count_median"] = int(owner_na_new.sum())
df.loc[owner_na_new, "owner_count_median"] = 0.0
print("Valori 'non applicabili' posti a 0:", structural_fill)
print(f"Righe: {len(df)} (nessuna eliminata) | colonne: {df.shape[1]}")

### 6.4 Imputazione dei pochi mancanti residui
Le variabili con meno dell'1% di NaN (dimensioni, consumi, potenza, gli `owner_count` restanti di venditori di usato, prezzi ambigui, valori resi NaN in §5) si imputano con la **mediana del gruppo `dominant_body_type`**. Dimensioni e consumi dipendono dal tipo di carrozzeria (un pickup non ha le misure di una berlina), e la mediana è robusta agli outlier.

In [ ]:
small_na_cols = [c for c in NUM if c in df.columns and c != "engine_displacement_median" and df[c].isna().any()]
imputed_counts = {}
for col in small_na_cols:
    n_na = int(df[col].isna().sum())
    df[col] = df.groupby("dominant_body_type")[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())      # rete di sicurezza per gruppi senza valori
    imputed_counts[col] = n_na
print("Valori imputati con la mediana per body type:", imputed_counts)

### 6.5 `engine_displacement_median` (24,5% di NaN): confronto di 4 strategie
Come nel lab Titanic con `Age`, si confrontano 4 strategie:
1. **mediana globale**;
2. **mediana per `dominant_body_type`**;
3. **estrazione casuale** dalla distribuzione osservata (DM1.02 §8);
4. **regressione lineare** da `horsepower_median` e `fuel_tank_volume_median` (correlazioni ~0,7–0,8 con la cilindrata).

Due criteri:
- **accuratezza**: si nasconde il 20% dei valori *noti* e si misura l'errore (MAE, RMSE) nel ricostruirli;
- **forma della distribuzione**: media e deviazione standard della colonna dopo l'imputazione, e istogramma dei valori imputati rispetto a quelli osservati.

In [ ]:
ED = "engine_displacement_median"
REG_X = ["horsepower_median", "fuel_tank_volume_median"]
rng = np.random.default_rng(RANDOM_STATE)
known = df[df[ED].notna()]
holdout_idx = known.sample(frac=0.2, random_state=RANDOM_STATE).index
train = known.drop(index=holdout_idx)


def impute_engine(strategy, fit_df, target_idx):
    """Valori imputati per le righe target_idx, con la strategia stimata solo su fit_df."""
    if strategy == "mediana globale":
        return pd.Series(fit_df[ED].median(), index=target_idx)
    if strategy == "mediana per body type":
        med = fit_df.groupby("dominant_body_type")[ED].median()
        return df.loc[target_idx, "dominant_body_type"].map(med).fillna(fit_df[ED].median())
    if strategy == "estrazione casuale":
        return pd.Series(rng.choice(fit_df[ED].values, size=len(target_idx)), index=target_idx)
    if strategy == "regressione (hp + serbatoio)":
        model = LinearRegression().fit(fit_df[REG_X], fit_df[ED])
        pred = model.predict(df.loc[target_idx, REG_X])
        return pd.Series(np.clip(pred, fit_df[ED].min(), fit_df[ED].max()), index=target_idx)
    raise ValueError(strategy)


STRATEGIES = ["mediana globale", "mediana per body type", "estrazione casuale", "regressione (hp + serbatoio)"]
missing_idx = df.index[df[ED].isna()]
results, imputed_versions = [], {}
for strat in STRATEGIES:
    pred_hold = impute_engine(strat, train, holdout_idx)
    err = pred_hold - known.loc[holdout_idx, ED]
    filled = df[ED].copy()
    filled.loc[missing_idx] = impute_engine(strat, known, missing_idx)
    imputed_versions[strat] = filled
    results.append({"strategia": strat, "MAE (cc)": err.abs().mean(), "RMSE (cc)": np.sqrt((err ** 2).mean()),
                    "media dopo": filled.mean(), "std dopo": filled.std()})
imputation_table = pd.DataFrame(results).set_index("strategia")
imputation_table.loc["solo valori osservati"] = [np.nan, np.nan, known[ED].mean(), known[ED].std()]
display(imputation_table.round(1))
save_table(imputation_table.round(1), "tab_imputazione_cilindrata")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6), sharey=True)
bins = np.histogram_bin_edges(known[ED], bins=40)
for ax, strat in zip(axes, STRATEGIES):
    ax.hist(known[ED], bins=bins, density=True, color="#b7d3f6", edgecolor=SURFACE, label="osservati")
    ax.hist(imputed_versions[strat].loc[missing_idx], bins=bins, density=True, histtype="step", lw=2,
            color=ORANGE, label="imputati")
    row = imputation_table.loc[strat]
    ax.set_title(f"{strat}\nMAE = {row['MAE (cc)']:.0f} cc · std = {row['std dopo']:.0f}", fontsize=9)
    ax.set_xlabel("cilindrata (cc)")
axes[0].set_ylabel("densità")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("engine_displacement_median: valori osservati vs imputati", y=1.04)
savefig(fig, "fig06_2_imputazione_cilindrata")
plt.show()

In [ ]:
# Scelta: la strategia con l'errore più basso sul 20% nascosto
best_strategy = imputation_table["MAE (cc)"].idxmin()
print("Strategia scelta:", best_strategy)
df["engine_displacement_missing"] = df[ED].isna().astype(int)   # traccia di quali valori sono stimati
df[ED] = imputed_versions[best_strategy]

# Categoriche: 'unknown' rarissimi → moda; il colore 'unknown' (11%) resta una categoria
for col in ["dominant_fuel_type", "dominant_transmission"]:
    mode = df.loc[df[col] != "unknown", col].mode()[0]
    n = int((df[col] == "unknown").sum())
    df[col] = df[col].replace("unknown", mode)
    print(f"{col}: {n} 'unknown' → '{mode}'")

print("NaN residui:", int(df.isna().sum().sum()))

**Commento per il report.**
- **Il fatto di mancare non porta informazione** né per `engine_displacement_median` né per `dominant_wheel_system`: le mediane delle altre variabili e la distribuzione del rating sono praticamente uguali tra righe con e senza mancante (V di Cramér con il target ≈ 0,01, p > 0,2). I mancanti sono quindi "casuali" (un problema di raccolta, non una caratteristica dei venditori).
- `dominant_wheel_system` (76% NaN) è stata **eliminata**: imputare tre quarti dei valori vorrebbe dire inventare la variabile, e il test precedente mostra che eliminandola non si perde un'informazione nascosta.
- I 27 NaN **in blocco** sono **mancanti strutturali** di venditori di solo nuovo ("non applicabile"): sono stati posti a 0 invece di eliminare le righe, che avrebbe cancellato quasi tutta la categoria `new_only`.
- `engine_displacement_median` (24,5% NaN): i mancanti non si concentrano tra le auto elettriche (sono quasi tutti "gasoline"), quindi non dipendono dal fatto che la cilindrata non esista. La **regressione** su potenza e serbatoio ricostruisce i valori nascosti con l'errore più basso (MAE ≈ 330 cc contro ≈ 620 della mediana globale e ≈ 520 della mediana per body type) e conserva la variabilità meglio della mediana globale, che crea un picco artificiale e riduce la deviazione standard (lo stesso effetto visto nel lab Titanic). La colonna `engine_displacement_missing` serve solo a sapere quali valori sono stimati, non come predittore (il mancante non è informativo).
- I pochi `unknown` di alimentazione e cambio sono stati sostituiti con la moda; il colore `unknown` (~11%) resta una categoria, perché non c'è un modo sensato di ricostruirlo.
- **Data leakage**: qui imputazioni e statistiche sono stimate su tutto il dataset, il che va bene per l'esplorazione. In classificazione andranno ristimate sul solo training set.

## 7. Outlier, trasformazioni e nuove feature

### 7.1 Outlier univariati (regola k·IQR)

In [ ]:
NUM_NOW = [c for c in NUM if c in df.columns]
out_before = dp.iqr_outlier_table(df, NUM_NOW)
display(out_before.sort_values("%_out_1.5IQR", ascending=False))

print("Mega-dealer (top 5 per n_listings):")
display(df.nlargest(5, "n_listings")[["n_listings", "n_listing_days", "franchise_status", "pct_is_new", TARGET]])
print("Auto d'epoca (year_median < 1990):", int((df["year_median"] < 1990).sum()),
      "| inventari prevalentemente elettrici:", int((df["dominant_fuel_type"] == "electric").sum()))

**Commento.**
- Per le variabili molto asimmetriche la regola 1.5·IQR segnala centinaia di "outlier" (es. `n_listings` ~4,6%, `daysonmarket_median` ~8%). Non sono errori: è la **coda di una distribuzione a coda lunga**, e venditori molto grandi o auto d'epoca esistono davvero. Eliminarli toglierebbe dati veri. La scelta è **tenerli e ridurne l'effetto con il logaritmo** (§7.2).
- **Limite della regola IQR**: per `pct_salvage` e `pct_theft_title` più del 75% dei valori è 0, quindi Q1 = Q3 = 0 e IQR = 0. **Ogni valore diverso da zero** risulta "outlier", e la regola non ha senso. Per queste variabili si binarizza (§7.3).
- Resta da ricordare per il Modulo 2 che K-means è sensibile agli outlier e che DBSCAN li potrà etichettare come rumore.

### 7.2 Trasformazione logaritmica
`log1p(x) = log(1 + x)` comprime la coda destra e gestisce gli zeri. Si applica a **conteggi, durate e importi** con |skew| > 2. Si escludono `pct_salvage` e `pct_theft_title`, che vengono binarizzate (§7.3), e `city_fuel_economy_median`, la cui asimmetria è dovuta a pochi veicoli elettrici. Per confronto si riporta anche la **radice quadrata** (DM1.02 §12), una trasformazione più "morbida".

In [ ]:
LOG_COLS = ["n_listings", "listing_span_days", "daysonmarket_median", "price_median", "savings_amount_median"]
log_df = np.log1p(df[LOG_COLS])
log_table = pd.DataFrame({
    "skew prima": df[LOG_COLS].skew(),
    "skew dopo log1p": log_df.skew(),
    "skew dopo sqrt": np.sqrt(df[LOG_COLS]).skew(),
    "% outlier 1.5·IQR prima": out_before.loc[LOG_COLS, "%_out_1.5IQR"],
    "% outlier 1.5·IQR dopo log1p": dp.iqr_outlier_table(log_df, LOG_COLS)["%_out_1.5IQR"],
})
display(log_table.round(2))
save_table(log_table.round(2), "tab_trasformazione_log")

fig, axes = plt.subplots(2, 4, figsize=(17, 6))
for j, col in enumerate(["n_listings", "daysonmarket_median", "price_median", "listing_span_days"]):
    axes[0, j].hist(df[col], bins=50, color="#b7d3f6", edgecolor=SURFACE)
    axes[0, j].set_title(f"{col}\nskew = {df[col].skew():.2f}", fontsize=9)
    axes[1, j].hist(log_df[col], bins=50, color=BLUE, edgecolor=SURFACE)
    axes[1, j].set_title(f"log1p({col})\nskew = {log_df[col].skew():.2f}", fontsize=9)
axes[0, 0].set_ylabel("originale")
axes[1, 0].set_ylabel("log1p")
fig.suptitle("Effetto della trasformazione logaritmica", y=1.0)
fig.tight_layout()
savefig(fig, "fig07_1_log_prima_dopo")
plt.show()

**Commento per il report.** Il logaritmo porta l'asimmetria vicino a 0 per `n_listings`, `price_median`, `daysonmarket_median` e `savings_amount_median`, e riduce molto la quota di outlier IQR (per `n_listings` da ~4,6% a quasi 0). Due limiti da dire:
- per `listing_span_days` il log **corregge troppo** (asimmetria da positiva a negativa, per i venditori con span vicino a 0) e la radice quadrata dà un'asimmetria più vicina a 0;
- per `savings_amount_median` il ~37% di zeri resta un picco separato anche dopo il log.

Per semplicità e coerenza si usa `log1p` per tutte (è monotona, quindi non cambia l'ordine dei valori). La radice quadrata per `listing_span_days` resta un'alternativa da provare nel clustering.

### 7.3 Categorie rare e binarizzazione
- `dominant_body_type`: le 6 categorie sotto il 2% vengono raggruppate in modo **semantico**: `coupe_convertible` (sportive) e `other` (van, minivan, hatchback, wagon). Categorie con pochi casi creano rumore nel one-hot e regole poco supportate nel pattern mining.
- `dominant_listing_color`: brown, green, gold e teal (< 0,3% insieme) → `other`.
- `pct_salvage` e `pct_theft_title` sono 0 per l'85–91% dei venditori → binarizzate in `has_salvage` e `has_theft_title`: l'informazione utile è "ne ha o no".

In [ ]:
body_map = {"coupe": "coupe_convertible", "convertible": "coupe_convertible",
            "van": "other", "minivan": "other", "hatchback": "other", "wagon": "other"}
df["dominant_body_type"] = df["dominant_body_type"].replace(body_map)
df["dominant_listing_color"] = df["dominant_listing_color"].replace(
    {c: "other" for c in ["brown", "green", "gold", "teal"]})
df["has_salvage"] = (df["pct_salvage"] > 0).astype(int)
df["has_theft_title"] = (df["pct_theft_title"] > 0).astype(int)
display(df["dominant_body_type"].value_counts(normalize=True).round(3).to_frame())
display(df[["has_salvage", "has_theft_title"]].mean().round(3).to_frame("quota di 1"))

### 7.4 Feature engineering
| Nuova feature | Definizione | Perché |
|---|---|---|
| `listings_per_day` | `n_listings / n_listing_days` | intensità dell'attività, separata dalla dimensione; sostituisce la coppia ridondante (ρ ≈ 0,96) |
| `combined_fuel_economy` | `0.55·city + 0.45·highway` (pesi EPA) | sostituisce due variabili con ρ ≈ 0,94 |
| `discount_ratio` | `savings_amount / price` | sconto relativo, confrontabile tra fasce di prezzo diverse |

In [ ]:
df["listings_per_day"] = df["n_listings"] / df["n_listing_days"]
df["combined_fuel_economy"] = 0.55 * df["city_fuel_economy_median"] + 0.45 * df["highway_fuel_economy_median"]
df["discount_ratio"] = df["savings_amount_median"] / df["price_median"]
NEW_FEATURES = ["listings_per_day", "combined_fuel_economy", "discount_ratio"]

target_code = df[TARGET].map({c: i for i, c in enumerate(ORDER)})
display(pd.DataFrame({
    "skew": df[NEW_FEATURES].skew(),
    "rho con target": [spearmanr(df[c], target_code)[0] for c in NEW_FEATURES],
}).round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, col in zip(axes, NEW_FEATURES):
    y = np.log1p(df[col]) if col == "listings_per_day" else df[col]
    sns.boxplot(x=df[TARGET], y=y, order=ORDER, hue=df[TARGET], hue_order=ORDER, palette=CLASS_COLORS,
                showfliers=False, ax=ax, legend=False)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("log1p" if col == "listings_per_day" else "")
fig.suptitle("Nuove feature per classe di rating", y=1.03)
fig.tight_layout()
savefig(fig, "fig07_2_nuove_feature")
plt.show()

## 8. Correlazioni ed eliminazione di variabili

### 8.1 Matrice di correlazione
**Spearman** come misura principale (relazioni monotone, robusta a outlier e asimmetrie); **Pearson** per confronto: dove i due valori differiscono molto la relazione non è lineare oppure è guidata da outlier.

In [ ]:
CORR_COLS = [c for c in NUM if c in df.columns] + NEW_FEATURES
spear = df[CORR_COLS].corr(method="spearman")
pears = df[CORR_COLS].corr(method="pearson")

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(spear, dtype=bool))
annot = spear.map(lambda v: f"{v:.2f}" if abs(v) >= 0.7 else "")
sns.heatmap(spear, mask=mask, cmap=DIVERGING, vmin=-1, vmax=1, center=0, annot=annot, fmt="",
            annot_kws={"fontsize": 7}, linewidths=0.5, linecolor=SURFACE,
            cbar_kws={"label": "ρ di Spearman", "shrink": 0.7}, ax=ax)
ax.set_title("Correlazione di Spearman (annotati i valori con |ρ| ≥ 0.7)")
ax.grid(False)
savefig(fig, "fig08_1_heatmap_spearman")
plt.show()

In [ ]:
iu = np.triu_indices_from(spear, k=1)
pairs = pd.DataFrame({
    "var_1": spear.columns[iu[0]], "var_2": spear.columns[iu[1]],
    "spearman": spear.values[iu], "pearson": pears.values[iu],
})
pairs["|diff|"] = (pairs["spearman"] - pairs["pearson"]).abs()
high_pairs = pairs[pairs["spearman"].abs() >= 0.7].sort_values("spearman", key=abs, ascending=False)
display(high_pairs.round(3).reset_index(drop=True))
save_table(high_pairs.round(3).reset_index(drop=True), "tab_coppie_correlate")

print("Coppie dove Pearson e Spearman differiscono di più:")
display(pairs.sort_values("|diff|", ascending=False).head(8).round(3).reset_index(drop=True))

### 8.2 Associazione tra categoriche (V di Cramér)
Per le variabili categoriche la correlazione non è definita: si usa la **V di Cramér** (da 0 = indipendenza a 1 = associazione perfetta), calcolata dal χ² della tabella di contingenza.

In [ ]:
CAT_NOW = ["dominant_body_type", "dominant_fuel_type", "dominant_transmission", "dominant_listing_color",
           "franchise_status", "new_used_profile", "has_salvage", "has_theft_title", TARGET]
cv = dp.cramers_v_matrix(df, CAT_NOW)
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cv, cmap=SEQUENTIAL, vmin=0, vmax=1, annot=True, fmt=".2f", annot_kws={"fontsize": 8},
            linewidths=0.5, linecolor=SURFACE, cbar_kws={"label": "V di Cramér"}, ax=ax)
ax.set_title("Associazione tra variabili categoriche")
ax.grid(False)
savefig(fig, "fig08_2_cramers_v")
plt.show()
display(pd.crosstab(df["franchise_status"], df["new_used_profile"]))

### 8.3 Alternativa: PCA sul blocco "dimensioni del veicolo"
Dimensioni, serbatoio, potenza e cilindrata sono correlate tra loro (0,6–0,85). Invece di eliminarne alcune, si possono riassumere con una PCA (DM1.02 §5). Vantaggio: la ridondanza si riduce senza scegliere a mano cosa scartare. Svantaggio: le componenti sono meno interpretabili.

In [ ]:
SIZE_COLS = ["length_median", "width_median", "height_median", "fuel_tank_volume_median",
             "horsepower_median", "engine_displacement_median"]
pca = PCA().fit(StandardScaler().fit_transform(df[SIZE_COLS]))
evr = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(len(SIZE_COLS))])
loadings = pd.DataFrame(pca.components_[:2].T, index=SIZE_COLS, columns=["PC1", "PC2"])
display(pd.DataFrame({"varianza spiegata": evr, "cumulata": evr.cumsum()}).round(3))
display(loadings.round(3))

fig, ax = plt.subplots(figsize=(6, 3.3))
ax.bar(evr.index, evr.values, color=BLUE, edgecolor=SURFACE)
ax.plot(evr.index, evr.cumsum().values, color=INK, marker="o", lw=1.5, label="cumulata")
ax.set_ylim(0, 1.05)
ax.set_ylabel("quota di varianza")
ax.legend(frameon=False)
ax.set_title("PCA sul blocco dimensioni/motore")
savefig(fig, "fig08_3_pca_dimensioni")
plt.show()

### 8.4 Regola di decisione e variabili eliminate
**Regola dichiarata in anticipo**:
- |ρ| ≥ 0,9 → si tiene una sola variabile (la più interpretabile, con meno valori imputati, più legata al target) oppure si fondono le due in una nuova feature;
- 0,7 ≤ |ρ| < 0,9 → si valuta caso per caso (spesso si tengono entrambe: sono correlate ma non duplicate);
- variabili **quasi costanti** (> 95% stesso valore) o **derivate** da altre → si eliminano dal set di modellazione.

In [ ]:
DROP = {
    "year_median": "ρ ≈ −0.98 con vehicle_age_median (age = 2020 − year): ridondante per costruzione",
    "n_listing_days": "ρ ≈ 0.96 con n_listings; l'informazione residua resta in listings_per_day",
    "city_fuel_economy_median": "ρ ≈ 0.94 con highway: entrambe sostituite da combined_fuel_economy",
    "highway_fuel_economy_median": "ρ ≈ 0.94 con city: entrambe sostituite da combined_fuel_economy",
    "pct_salvage": "~91% zeri: sostituita da has_salvage",
    "pct_theft_title": "~85% zeri: sostituita da has_theft_title",
    "dominant_fuel_type": "quasi costante (gasoline ~99%)",
    "dominant_wheel_system": "76% di NaN (già eliminata in §6)",
}
KEEP_BUT_FLAG = {
    "new_used_profile": "determinata da pct_is_new: esclusa dai modelli, tenuta per il pattern mining",
    "engine_displacement_median": "24.5% imputato e ridondante con hp/serbatoio: esclusa dal clustering",
    "savings_amount_median": "ρ ≈ 0.91 con discount_ratio (versione relativa al prezzo): esclusa dal clustering",
    "dominant_transmission": "quasi costante (automatico ~94%): tenuta solo nel dataset pulito",
}
display(pd.DataFrame({"motivo": DROP}))
display(pd.DataFrame({"motivo": KEEP_BUT_FLAG}))

**Commento per il report (§8).**
- Le correlazioni forti formano **blocchi interpretabili**: (i) *età/uso/nuovo* (`vehicle_age`, `year`, `mileage`, `owner_count`, `pct_is_new`, `savings`); (ii) *dimensione del veicolo* (dimensioni, serbatoio, potenza, cilindrata, consumi con segno negativo); (iii) *attività del venditore* (`n_listings`, `n_listing_days`).
- In diverse coppie Spearman è molto più alto di Pearson (es. `price`–`year`: 0,82 contro 0,55): la relazione è **monotona ma non lineare** e risente della coda di prezzi alti. È un'ulteriore ragione per usare il log su prezzo e conteggi.
- Tra le categoriche, `franchise_status` e `new_used_profile` sono fortemente associate (V alta: i franchise vendono anche nuovo, gli indipendenti quasi solo usato). Tutte le categoriche hanno un'associazione debole con il target.
- Anche le nuove feature vanno controllate con la stessa regola: `discount_ratio` ha ρ ≈ 0,91 con `savings_amount_median`. Nel dataset per il clustering si tiene solo `discount_ratio`, perché è relativo al prezzo e quindi confrontabile tra venditori di fasce diverse.
- La PCA sul blocco dimensioni/motore spiega il **71%** della varianza con la sola PC1, che ha pesi simili e positivi su tutte e sei le variabili ("taglia/potenza del veicolo"). È un'alternativa da valutare nel clustering, se le variabili originali risultano troppo ridondanti.

## 9. Dataset finali, "contratto" e riepilogo delle decisioni

Si salvano due file in `data/processed/` (stesse righe, indice `sp_id`):
1. **`seller_clean.csv`**: dataset ripulito, **non trasformato né scalato**, con target. Base per classificazione, regressione e (dopo la discretizzazione) pattern mining.
2. **`seller_clustering_base.csv`**: solo variabili numeriche non ridondanti, log dove serve, **z-score**, **senza target** (il target serve solo dopo, per interpretare i cluster). La scelta finale degli attributi per il clustering si fa nel Modulo 2.

In [ ]:
df_clean = df.drop(columns=[c for c in DROP if c in df.columns])

CLUSTER_COLS = ["n_listings", "listings_per_day", "listing_span_days", "daysonmarket_median",
                "price_median", "discount_ratio", "mileage_median",
                "vehicle_age_median", "owner_count_median", "horsepower_median", "combined_fuel_economy",
                "back_legroom_median", "front_legroom_median", "fuel_tank_volume_median",
                "height_median", "length_median", "width_median", "pct_isCab", "pct_is_new"]
CLUSTER_LOG = ["n_listings", "listings_per_day", "listing_span_days", "daysonmarket_median", "price_median"]
df_cluster = df_clean[CLUSTER_COLS].copy()
df_cluster[CLUSTER_LOG] = np.log1p(df_cluster[CLUSTER_LOG])
df_cluster = pd.DataFrame(StandardScaler().fit_transform(df_cluster), index=df_cluster.index,
                          columns=[f"log1p_{c}" if c in CLUSTER_LOG else c for c in CLUSTER_COLS])

# Controlli automatici prima di salvare
assert df_clean.isna().sum().sum() == 0, "NaN residui in df_clean"
assert df_cluster.isna().sum().sum() == 0, "NaN residui in df_cluster"
assert df_clean.index.equals(df_cluster.index), "righe diverse tra i due file"
assert TARGET not in df_cluster.columns
assert np.allclose(df_cluster.mean(), 0, atol=1e-8) and np.allclose(df_cluster.std(ddof=0), 1, atol=1e-6)

df_clean.to_csv(OUT_DIR / "seller_clean.csv")
df_cluster.to_csv(OUT_DIR / "seller_clustering_base.csv")
print("seller_clean.csv:", df_clean.shape, "| seller_clustering_base.csv:", df_cluster.shape)
print(df_clean.dtypes.value_counts())

In [ ]:
# "Contratto" del dataset: cosa c'è in ogni file e che trattamento ha subito
imputed_all = {c: imputed_counts.get(c, 0) + structural_fill.get(c, 0)
               for c in set(imputed_counts) | set(structural_fill)}
imputed_all[ED] = len(missing_idx)
contract = pd.DataFrame(index=df_clean.columns)
contract["in clustering_base"] = [("sì" if c in CLUSTER_COLS else "no") for c in contract.index]
contract["trasformazione (clustering)"] = [("log1p + z-score" if c in CLUSTER_LOG else
                                            "z-score" if c in CLUSTER_COLS else "—") for c in contract.index]
contract["n imputati"] = [imputed_all.get(c, 0) for c in contract.index]
contract["nota"] = [KEEP_BUT_FLAG.get(c, "nuova feature" if c in NEW_FEATURES + ["has_salvage", "has_theft_title",
                    "engine_displacement_missing"] else "") for c in contract.index]
display(contract)
save_table(contract, "tab_contratto_dataset")

In [ ]:
# Tabella riassuntiva delle decisioni (da mettere a fine sezione nel report)
decisions = pd.DataFrame([
    ("7 colonne numeriche", "valori corrotti (separatore migliaia)", "ricostruzione univoca, altrimenti NaN", "99.7% recuperati senza distorsioni"),
    ("sp_id", "letto come float, è un ID", "intero, usato come indice", "non è un predittore"),
    ("dominant_transmission", "codici opachi (a, m)", "rinominati", "leggibilità"),
    ("legroom, consumi, mileage", "valori impossibili/incoerenti", "→ NaN, poi imputati", "errori, non valori veri"),
    ("year < 1990, EV, van alti, mega-dealer", "outlier", "tenuti", "valori veri e interpretabili"),
    ("dominant_wheel_system", "76% NaN", "eliminata", "imputazione non credibile"),
    ("pct_isCab/salvage/theft, owner_count", "NaN in blocco per venditori di solo nuovo", "→ 0 ('non applicabile')", "un'auto nuova non ha proprietari né precedenti; righe tenute"),
    ("engine_displacement_median", "24.5% NaN", f"{best_strategy} + flag", "errore minimo sul 20% nascosto"),
    ("NaN residui < 1%", "sparsi", "mediana per body type", "dimensioni dipendono dalla carrozzeria"),
    ("fuel/transmission 'unknown'", "pochissimi casi", "moda", "impatto nullo"),
    ("colore 'unknown'", "11%, mancante mascherato", "categoria a sé", "non ricostruibile"),
    ("conteggi, prezzi, durate", "forte asimmetria", "log1p (per clustering)", "skew e outlier ridotti"),
    ("body type, colore", "categorie rare", "raggruppate", "supporto troppo basso"),
    ("pct_salvage, pct_theft_title", "quasi sempre 0", "binarizzate", "conta la presenza"),
    ("year, n_listing_days, city/highway", "|ρ| ≥ 0.94", "eliminate / fuse", "ridondanti"),
    ("savings_amount_median", "ρ ≈ 0.91 con discount_ratio", "esclusa dal clustering", "ridondante"),
    ("dominant_fuel_type", "~99% gasoline", "eliminata", "quasi costante"),
    ("new_used_profile", "derivata da pct_is_new", "esclusa dai modelli", "ridondante"),
], columns=["variabile", "problema", "azione", "motivazione"]).set_index("variabile")
display(decisions)
save_table(decisions, "tab_riepilogo_decisioni")